## 1. Imports, policy, claims

In [1]:
# 1. Imports, policy, claims (no labels)
%load_ext autoreload
%autoreload 2

import re
from collections.abc import Callable
from datetime import date, timedelta
from datetime import date as Date
from typing import Any, Literal, get_args

import pandas as pd
from pydantic import (
    BaseModel,
    ConfigDict,
    Field,
    ValidationError,
    field_validator,
)

from expense_agent.data import load_claims
from expense_agent.paths import GENERATED_DIR, POLICY_PATH
from expense_agent.policy import cap_usage, load_policy

policy = load_policy(POLICY_PATH)
claims_df = load_claims(GENERATED_DIR)

print(f"policy v{policy['version']} claims={len(claims_df)}")
assert len(claims_df) == 330, "Expected the generated set (330 claims)"
assert "in_policy" not in claims_df.columns, "Label leak in claims_df"

policy v1.0 claims=330


## Typed input

In [2]:
# 2. Typed input (Claim) and output (Violation)
ViolationCode = Literal[
    "over_cap",
    "prohibited_category",
    "missing_receipt",
    "late_submission",
    "duplicate",
    "split_transaction",
    "personal_extension",
    "premium_class",
]


class Claim(BaseModel):
    """One submitted expense line item. Carries no labels."""

    model_config = ConfigDict(frozen=True, extra="ignore")

    report_id: str
    claim_id: str
    employee_id: str
    department: str
    submitted_date: date
    expense_date: date
    category: str
    merchant: str
    description: str
    business_purpose: str | None = None
    amount: float = Field(gt=0)
    city: str | None = None
    nights: int | None = Field(default=None, ge=0)
    attendees: int | None = Field(default=None, ge=0)
    fare_class: str | None = None
    flight_hours: float | None = None
    rental_class: str | None = None
    miles: int | None = Field(default=None, ge=0)
    preapproved: bool | None = None
    receipt_attached: bool

    @field_validator(
        "*", mode="before"
    )  # strip whitespace from all string fields
    @classmethod
    def blank_to_none(cls, value: object) -> object:
        """Turn pandas NaN/<NA> and empty strings into None."""
        if isinstance(value, str):
            return value.strip() or None
        if value is not None and pd.isna(value):
            return None
        return value


class Violation(BaseModel):
    """One broken rule on one claim, citing the policy section."""

    model_config = ConfigDict(frozen=True, extra="forbid")

    claim_id: str
    code: ViolationCode
    policy_ref: str = Field(pattern=r"^POL-\d+\.\d+$")
    detail: str = Field(min_length=1)


def to_claims(df: pd.DataFrame) -> list[Claim]:
    """Validate every row into a Claim; fail loudly on a bad row."""
    return [Claim.model_validate(row) for row in df.to_dict("records")]


claims = to_claims(claims_df)
print(f"validated {len(claims)} claims")

validated 330 claims


In [3]:
# 2b - checks
assert len(claims) == 330
assert len({c.claim_id for c in claims}) == 330, "claim_id not unique"

# Codes in code must match codes in config (single source of truth)
assert set(get_args(ViolationCode)) == set(policy["violation_codes"]), (
    "ViolationCode drifted from policy.yaml"
)

# A bad citation must be rejected
bad = {
    "claim_id": "X",
    "code": "over_cap",
    "policy_ref": "Pol-2.2",
    "detail": "test",
}
try:
    Violation(**bad)
except ValidationError as err:
    for error in err.errors():
        print(error["loc"], "->", error["msg"])
    print(f"rejected as expected: {err.error_count()} error(s)")
else:
    raise AssertionError("bad policy_ref was accepted")

('policy_ref',) -> String should match pattern '^POL-\d+\.\d+$'
rejected as expected: 1 error(s)


## 3 Single-claim rules

In [4]:
# 3. single-claim rules: late submission, missing receipt

Policy = dict[str, Any]
Rule = Callable[[Claim, Policy], Violation | None]


def check_late(claim: Claim, policy: Policy) -> Violation | None:
    """Flag claims submitted more than N days after the expense (POL-1.2)."""
    max_days = policy["submission"]["max_days_after_expense"]
    days = (claim.submitted_date - claim.expense_date).days
    if days > max_days:
        return Violation(
            claim_id=claim.claim_id,
            code="late_submission",
            policy_ref="POL-1.2",
            detail=f"submitted {days} days after expense (limit {max_days})",
        )
    return None


def check_receipt(claim: Claim, policy: Policy) -> Violation | None:
    """Flag items over the receipt threshold with no receipt (POL-1.3)."""
    if claim.category == "Mileage":
        return None  # mileage is miles x rate; there is no receipt
    threshold = policy["submission"]["receipt_required_over"]
    if claim.amount > threshold and not claim.receipt_attached:
        return Violation(
            claim_id=claim.claim_id,
            code="missing_receipt",
            policy_ref="POL-1.3",
            detail=(f"${claim.amount:,.2f} over ${threshold:,.2f} no receipt"),
        )
    return None


SINGLE_CLAIM_RULES: list[Rule] = [check_late, check_receipt]


found = [
    v
    for claim in claims
    for rule in SINGLE_CLAIM_RULES
    if (v := rule(claim, policy)) is not None
]
print(pd.Series([v.code for v in found]).value_counts().to_string())

missing_receipt    14
late_submission    11


In [5]:
# 3b -boundary tests
base = next(c for c in claims if c.category == "Meals" and c.receipt_attached)


def days_before_submit(n: int) -> Claim:
    """Copy of base with the expense dated n days before submission."""
    return base.model_copy(
        update={"expense_date": base.submitted_date - timedelta(days=n)}
    )


# Late: 60 days is on  tim, 61 is late (POL-1.2: "more than 60")
assert check_late(days_before_submit(60), policy) is None
assert check_late(days_before_submit(61), policy).code == "late_submission"

# Receipt: exactly $75 is fine, $75.01 without a receipt is not
at_limit = base.model_copy(update={"amount": 75.00, "receipt_attached": False})
over = base.model_copy(update={"amount": 75.01, "receipt_attached": False})
assert check_receipt(at_limit, policy) is None
assert check_receipt(over, policy).code == "missing_receipt"


# Mileage never needs a receipt
mileage = base.model_copy(
    update={"category": "Mileage", "amount": 500.0, "receipt_attached": False}
)
assert check_receipt(mileage, policy) is None

# Anchor to know numbers: 01b found exactly 11 late claims
n_late = sum(v.code == "late_submission" for v in found)
assert n_late == 11, f"expected 11 late claims, got {n_late}"
print("boundary tests passed")

boundary tests passed


## 4 cap rules 

In [6]:
# 4 cap rules (POL-2.2, 2.3, 2.4, 2.7, 2.9, 2.10)

CAP_POLICY_REFS: dict[str, str] = {
    "Lodging": "POL-2.2",
    "Meals": "POL-2.3",
    "Client Meals": "POL-2.4",
    "Conference": "POL-2.7",
    "Home Office": "POL-2.9",
    "Office Supplies": "POL-2.10",
}
CAP_UNITS: dict[str, str] = {"Lodging": "/night", "Client Meals": "/person"}


def check_cap(claim: Claim, policy: Policy) -> Violation | None:
    """Flag amounts over their category cap (POL-2.x by category)."""
    if claim.category not in policy["categories"]:
        return None  # not a policy category; the prohibited rule owns it
    usage = cap_usage(claim.model_dump(), policy)
    if usage is None:
        return None  # uncapped category, or no nights/attendees
    value, cap = usage
    if round(value, 2) > cap:
        unit = CAP_UNITS.get(claim.category, "")
        return Violation(
            claim_id=claim.claim_id,
            code="over_cap",
            policy_ref=CAP_POLICY_REFS[claim.category],
            detail=(
                f"{claim.category} ${value:,.2f}{unit} "
                f"over cap ${cap:,.2f}{unit}"
            ),
        )
    return None


def apply_rules(
    claims: list[Claim], rules: list[Rule], policy: Policy
) -> list[Violation]:
    """Run every rule on every claim; keep all violations found."""
    return [
        violation
        for claim in claims
        for rule in rules
        if (violation := rule(claim, policy)) is not None
    ]


SINGLE_CLAIM_RULES = [check_late, check_receipt, check_cap]
found = apply_rules(claims, SINGLE_CLAIM_RULES, policy)

print(pd.Series([v.code for v in found]).value_counts().to_string())
print(f"claims flagged: {len({v.claim_id for v in found})}")

over_cap           18
missing_receipt    14
late_submission    11
claims flagged: 43


In [7]:
# 4b cap boundary tests
lodging = next(c for c in claims if c.category == "Lodging" and c.nights)


def lodging_at(city: str, nightly: float, nights: int = 2) -> Claim:
    """Copy of a lodging claim at a given nightly rate and city."""
    return lodging.model_copy(
        update={"city": city, "nights": nights, "amount": nightly * nights}
    )


# Tier caps: $350 high-cost, $250 standard (POL-2.2)
assert check_cap(lodging_at("New York, NY", 350.00), policy) is None
assert check_cap(lodging_at("New York, NY", 350.01), policy).code == "over_cap"
assert check_cap(lodging_at("Austin, TX", 250.00), policy) is None
assert check_cap(lodging_at("Austin, TX", 300.00), policy).policy_ref == (
    "POL-2.2"
)


# Float safety: 7 nights at $249.99 must not flag
assert check_cap(lodging_at("Austin, TX", 249.99, nights=7), policy) is None

# Client Meals: zero or missing attendees must not crash
client_meal = next(c for c in claims if c.category == "Client Meals")
for n in (0, None):
    no_guests = client_meal.model_copy(update={"attendees": n})
    assert check_cap(no_guests, policy) is None

# Uncapped categories never flag
airfare = next(c for c in claims if c.category == "Airfare")
assert (
    check_cap(airfare.model_copy(update={"amount": 9_999.0}), policy) is None
)

# Sanity: every over_cap flag cities a POL-2.x section
assert all(
    v.policy_ref.startswith("POL-2.") for v in found if v.code == "over_cap"
)
print("cap tests passed")

cap tests passed


## 5: Class, vendor and mileage rules

In [8]:
# 5 fare/rental class, software vendor, mileage rate


def check_fare_class(claim: Claim, policy: Policy) -> Violation | None:
    """Flag premium airfare outside policy conditions (POL-2.1)."""
    if claim.category != "Airfare" or claim.fare_class is None:
        return None
    rules = policy["categories"]["Airfare"]
    if claim.fare_class in rules["allowed_fare_classes"]:
        return None
    min_hours = rules["business_class_min_flight_hours"]
    long_enough = (
        claim.flight_hours is not None and claim.flight_hours >= min_hours
    )
    if claim.fare_class == "Business" and long_enough and claim.preapproved:
        return None
    return Violation(
        claim_id=claim.claim_id,
        code="premium_class",
        policy_ref="POL-2.1",
        detail=(
            f"{claim.fare_class} on {claim.flight_hours}h flight, "
            f"preapproved={claim.preapproved} "
            f"(needs Business, >= {min_hours}h, preapproved)"
        ),
    )


def check_rental_class(claim: Claim, policy: Policy) -> Violation | None:
    """flag rental cars above the allowed classes (POL-2.5)."""
    if claim.category != "Ground Transport" or claim.rental_class is None:
        return None
    allowed = policy["categories"]["Ground Transport"][
        "allowed_rental_classes"
    ]
    if claim.rental_class in allowed:
        return None
    return Violation(
        claim_id=claim.claim_id,
        code="premium_class",
        policy_ref="POL-2.5",
        detail=f"rental class {claim.rental_class} not in {allowed}",
    )


def check_software_vendor(claim: Claim, policy: Policy) -> Violation | None:
    """Flag software bought from an unapproved vendor (POL-2.8)."""
    if claim.category != "Software":
        return None
    approved = policy["categories"]["Software"]["approved_vendors"]
    if claim.merchant in approved:
        return None
    return Violation(
        claim_id=claim.claim_id,
        code="prohibited_category",
        policy_ref="POL-2.8",
        detail=f"vendor {claim.merchant!r} not on approved list",
    )


def mileage_rate(expense_date: Date, policy: Policy) -> float:
    """Return the IRS mileage rate in force on a date (POL-2.6)."""
    for band in policy["categories"]["Mileage"]["rates"]:
        start = Date.fromisoformat(str(band["from"]))
        end = Date.fromisoformat(str(band["to"]))
        if start <= expense_date <= end:
            return band["rate"]
    raise ValueError(f"No mileage rate configured for {expense_date}")


def check_mileage(claim: Claim, policy: Policy) -> Violation | None:
    """Flag mileage claimed above miles x the dated rate (POL-2.6)."""
    if claim.category != "Mileage" or claim.miles is None:
        return None
    rate = mileage_rate(claim.expense_date, policy)
    allowed = round(claim.miles * rate, 2)
    if round(claim.amount, 2) > allowed:
        return Violation(
            claim_id=claim.claim_id,
            code="over_cap",
            policy_ref="POL-2.6",
            detail=(
                f"${claim.amount:,.2f} > {claim.miles} mi x ${rate}/mi "
                f"= ${allowed:,.2f}"
            ),
        )
    return None


SINGLE_CLAIM_RULES = [
    check_late,
    check_receipt,
    check_cap,
    check_fare_class,
    check_rental_class,
    check_mileage,
    check_software_vendor,
]
found = apply_rules(claims, SINGLE_CLAIM_RULES, policy)

print(pd.Series([v.code for v in found]).value_counts().to_string())
print(f"claims flagged: {len({v.claim_id for v in found})}")

over_cap           18
missing_receipt    14
late_submission    11
premium_class       6
claims flagged: 49


In [9]:
# 5b -class, vendor and mileage tests
flight = next(c for c in claims if c.category == "Airfare")


def fly(fare: str, hours: float | None, pre: bool) -> Claim:
    """Copy of an airfare claim with given fare, hours, preapproval."""
    return flight.model_copy(
        update={"fare_class": fare, "flight_hours": hours, "preapproved": pre}
    )


assert check_fare_class(fly("Economy", 2.0, False), policy) is None
assert check_fare_class(fly("Premium Economy", 9.0, False), policy) is None
assert check_fare_class(fly("Business", 6.0, True), policy) is None  # boundary
assert check_fare_class(fly("Business", 5.9, True), policy) is not None
assert check_fare_class(fly("Business", 8.0, False), policy) is not None
assert check_fare_class(fly("Business", None, True), policy) is not None
assert check_fare_class(fly("First", 12.0, True), policy) is not None

# Mileage rate changes on Jul 1, 2026
assert mileage_rate(Date(2026, 6, 30), policy) == 0.725
assert mileage_rate(Date(2026, 7, 1), policy) == 0.76

# Data-anchored expectations (generated set)
codes = pd.Series([v.code for v in found])
refs = pd.Series([v.policy_ref for v in found])
assert (refs == "POL-2.1").sum() == 6, "expected 6 premium airfare"
for ref in ("POL-2.5", "POL-2.6", "POL-2.8"):
    assert (refs == ref).sum() == 0, f"false positives under {ref}"
print("class, vendor and mileage tests passed")

class, vendor and mileage tests passed


## 6: Flags(prohibited hints, unapproved vendor)

In [10]:
# 6a - Flags: prohibited-keyword hints, unapproved vendors

FlagCode = Literal[
    "prohibited_hint",
    "unapproved_vendor",
    "high_value",
    "near_cap",
]


class Flag(BaseModel):
    """A reason for closer review (human or agent), not a denial."""

    model_config = ConfigDict(frozen=True, extra="forbid")

    claim_id: str
    code: FlagCode
    policy_ref: str = Field(pattern=r"^POL-\d+\.\d+$")
    detail: str = Field(min_length=1)


def compile_prohibited(policy: Policy) -> list[tuple[dict, re.Pattern[str]]]:
    """Build one case-insensitive, whole-word regex per POL-3 item."""
    compiled = []
    for item in policy["prohibited"]:
        words = "|".join(re.escape(word) for word in item["keywords"])
        pattern = re.compile(rf"\b(?:{words})s?\b", re.IGNORECASE)
        compiled.append((item, pattern))
    return compiled


PROHIBITED_PATTERNS = compile_prohibited(policy)


def check_prohibited_hint(claim: Claim, policy: Policy) -> Flag | None:
    """Hint at a possible POL-3 item from merchant/description text."""
    text = f"{claim.merchant} {claim.description}"
    for item, pattern in PROHIBITED_PATTERNS:
        alcohol = item["code"] == "ALCOHOL_NO_CLIENT"
        if alcohol and claim.category == "Client Meals":
            continue  # POL-2.4: alcohol is allowed at a client meal
        if match := pattern.search(text):
            return Flag(
                claim_id=claim.claim_id,
                code="prohibited_hint",
                policy_ref=item["policy_ref"],
                detail=f"{item['code']}: matched {match.group(0)!r}",
            )
    return None


def check_software_vendor(claim: Claim, policy: Policy) -> Flag | None:
    """Flag software from a vendor that needs IT approval (POL-2.8)."""
    if claim.category != "Software":
        return None
    approved = policy["categories"]["Software"]["approved_vendors"]
    if claim.merchant in approved:
        return None
    return Flag(
        claim_id=claim.claim_id,
        code="unapproved_vendor",
        policy_ref="POL-2.8",
        detail=f"vendor {claim.merchant!r} needs IT approval",
    )


FLAG_RULES = [check_prohibited_hint, check_software_vendor]
flags = apply_rules(claims, FLAG_RULES, policy)

hints = pd.DataFrame(
    [
        {
            "claim_id": f.claim_id,
            "category": c.category,
            "text": f"{c.merchant} | {c.description}",
            "detail": f.detail,
        }
        for f in flags
        for c in claims
        if c.claim_id == f.claim_id
    ]
)
print(pd.Series([f.code for f in flags]).value_counts().to_string())
hints

prohibited_hint    26


,claim_id,category,text,detail
0,R-E001-01-L3,Office Supplies,Amazon | Amazon gift card for team member birt...,GIFT_CARD: matched 'gift card'
1,R-E001-08-L4,Lodging,Marriott | In-room movies,WELLNESS_ENTERTAINMENT: matched 'In-room'
2,R-E004-06-L4,Office Supplies,Amazon | Amazon gift card for team member birt...,GIFT_CARD: matched 'gift card'
3,R-E005-02-L4,Lodging,Hotel Spa | Spa massage during trip,WELLNESS_ENTERTAINMENT: matched 'Spa'
4,R-E005-04-L2,Meals,Pinstripe Bar & Lounge | Drinks with coworkers...,ALCOHOL_NO_CLIENT: matched 'Bar'
5,R-E006-01-L1,Office Supplies,Amazon | Amazon gift card for team member birt...,GIFT_CARD: matched 'gift card'
6,R-E006-02-L2,Home Office,Best Buy | AirPods Pro for personal commute,PERSONAL_ITEM: matched 'personal'
7,R-E006-03-L3,Ground Transport,City Parking Enforcement | Parking ticket near...,FINE: matched 'Parking ticket'
8,R-E006-03-L4,Airfare,Delta Air Lines | Airfare for spouse to join trip,FAMILY_TRAVEL: matched 'spouse'
9,R-E006-10-L2,Ground Transport,City Parking Enforcement | Parking ticket near...,FINE: matched 'Parking ticket'


In [12]:
# 6b hint tests
meal = next(c for c in claims if c.category == "Meals")
client = next(c for c in claims if c.category == "Client Meals")


def says(base: Claim, merchant: str, description: str) -> Claim:
    """Copy of a claim with new merchant and description text."""
    return base.model_copy(
        update={"merchant": merchant, "description": description}
    )


hit = check_prohibited_hint(says(meal, "Corner Bar", "Team drinks"), policy)
assert hit is not None and hit.policy_ref == "POL-3.1"
assert (
    check_prohibited_hint(says(client, "Corner Bar", "Wine w/ Acme"), policy)
    is None
)
assert (
    check_prohibited_hint(says(meal, "Delta", "Airline ticket"), policy)
    is None
)
assert (
    check_prohibited_hint(says(meal, "Hotel", "In-room movies"), policy)
    is not None
)
assert (
    check_prohibited_hint(says(meal, "Cafe", "Workspace Lunch"), policy)
    is None
)


# Known false positive, documented on purpose: a hint, not a denial
fp = check_prohibited_hint(says(client, "Le Bernardin", "Fine dining"), policy)
print(f"known false positive: {fp.detail if fp else 'none'}")
print("hint tests passed")

known false positive: FINE: matched 'Fine'
hint tests passed
